# Benchmark analysis for `run_analysis.py`

This notebook reads the CSV files generated by `run_analysis.py`:

- `results/analysis_full_layers.csv`
- `results/analysis_timeseries.csv`


In [ ]:
from pathlib import Path
import os
import re
import inspect
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

sns.set_theme(style='ticks', context='notebook')
plt.rcParams.update({
    'font.size': 14.5,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'figure.dpi': 110,
})

NUM_EPISODES = 300
BARPLOT_KW = {'errorbar': None} if 'errorbar' in inspect.signature(sns.barplot).parameters else {'ci': None}

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = []
    candidates.extend([start, *start.parents])
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'run_analysis.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project folder. Set PROJECT_ROOT manually.')

PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results'
FULL_CSV = RESULTS_DIR / 'analysis_full_layers.csv'
TS_CSV = RESULTS_DIR / 'analysis_timeseries.csv'
LLM_MODELS = [m.strip() for m in os.getenv('LLM_MODELS', 'gpt-4o-mini,gpt-5-mini,gpt-5-nano').split(',') if m.strip()]
LLM_MODEL_LABELS = {
    'gpt-4o-mini': 'GPT-4o mini',
    'gpt-5-mini': 'GPT-5 mini',
    'gpt-5-nano': 'GPT-5 nano',
}
def model_slug(model):
    return re.sub(r'[^A-Za-z0-9._-]+', '-', model).strip('-._') or 'unknown-model'
LLM_FILES = {model: {
    'Goal': (RESULTS_DIR / f'analysis_llm_full_layers_{model_slug(model)}.csv', RESULTS_DIR / f'analysis_llm_timeseries_{model_slug(model)}.csv'),
    'NoGoal': (RESULTS_DIR / f'analysis_llm_full_layers_no_goal_{model_slug(model)}.csv', RESULTS_DIR / f'analysis_llm_timeseries_no_goal_{model_slug(model)}.csv'),
} for model in LLM_MODELS}

print(f'Project root: {PROJECT_ROOT}')
print(f'Summary CSV: {FULL_CSV}')
print(f'Timeseries CSV: {TS_CSV}')
for model, variants in LLM_FILES.items():
    print(f'{LLM_MODEL_LABELS.get(model, model)}:')
    for treatment, (summary_path, ts_path) in variants.items():
        print(f'  {treatment}: {summary_path.name} | {ts_path.name}')


In [ ]:
def _to_numeric(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

SUMMARY_METRICS = [
    ('avg_last50_global', 'Mean population, last 50 steps'),
    ('avg_steps_alive', 'Mean survival steps'),
    ('extinction_rate', 'Extinction rate'),
]

BEHAVIORAL_SUMMARY_METRICS = [
    ('avg_movement_frequency', 'Movement frequency'),
    ('avg_harvested_food', 'Harvested food per agent-step'),
    ('avg_reproduction_events', 'Reproduction events per episode'),
    ('avg_action_entropy', 'Action entropy'),
    ('avg_policy_confidence', 'Policy confidence'),
    ('avg_reward', 'Average reward per agent-step'),
]

TIMESERIES_METRICS = [
    ('Population among alive episodes', 'avg_pop_step'),
    ('Global population average', 'global_avg_pop'),
    ('Survival rate', 'survival_rate'),
    ('Rule propensity alpha', 'avg_prop_step'),
    ('Gene beta (rule agents)', 'avg_beta_step'),
    ('Movement frequency', 'avg_move_freq_step'),
    ('Harvested food per agent-step', 'avg_harvested_food_step'),
    ('Births per step', 'avg_births_step'),
    ('Action entropy', 'avg_entropy_step'),
    ('Policy confidence', 'avg_confidence_step'),
    ('Average reward per agent-step', 'avg_reward_step'),
]

NEW_SUMMARY_COLUMNS = [
    'avg_movement_frequency', 'std_movement_frequency',
    'avg_harvested_food', 'std_harvested_food',
    'avg_reproduction_events', 'std_reproduction_events',
    'avg_action_entropy', 'std_action_entropy',
    'avg_policy_confidence', 'std_policy_confidence',
    'avg_reward', 'std_reward',
]

NEW_TIMESERIES_COLUMNS = [
    'avg_move_freq_step',
    'avg_harvested_food_step',
    'avg_births_step',
    'avg_entropy_step',
    'avg_confidence_step',
    'avg_reward_step',
]

def parse_mode(mode):
    mode = str(mode)
    if mode == 'Random':
        return pd.Series({
            'family': 'Random',
            'training': 'None',
            'adaptation': 'Random',
            'short_mode': 'Random',
        })

    if mode.startswith('Rule-'):
        left, adaptation = mode.split('/', 1)
        init = left.replace('Rule-', '')
        return pd.Series({
            'family': 'Rule',
            'training': init,
            'adaptation': adaptation,
            'short_mode': mode,
        })

    if mode.startswith('MLP-'):
        left, adaptation = mode.split('/', 1)
        training = left.replace('MLP-', '')
        return pd.Series({
            'family': 'MLP',
            'training': training,
            'adaptation': adaptation,
            'short_mode': mode,
        })

    if mode == 'LLM' or mode.startswith('LLM-'):
        return pd.Series({
            'family': 'LLM',
            'training': 'LLM',
            'adaptation': 'Policy',
            'short_mode': mode,
        })

    return pd.Series({
        'family': 'Other',
        'training': 'Other',
        'adaptation': 'Other',
        'short_mode': mode,
    })

def load_summary(path=FULL_CSV):
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} not found. Run run_analysis.py first.')

    df = pd.read_csv(path)
    if Path(path).resolve() == FULL_CSV.resolve():
        llm_frames = []
        for model in LLM_MODELS:
            for treatment, (summary_path, _) in LLM_FILES[model].items():
                if not summary_path.exists():
                    print(f'LLM summary missing: {summary_path.name}')
                    continue
                llm_part = pd.read_csv(summary_path)
                llm_part['mode'] = f'LLM-{model}-{treatment}'
                llm_frames.append(llm_part)
                print(f'LLM summary appended: {summary_path}')
        if llm_frames:
            df = pd.concat([df, *llm_frames], ignore_index=True)
    df['mode'] = df['mode'].replace({'Random_Random': 'Random'})

    numeric_cols = [
        'layer',
        'avg_final_global', 'avg_final_survived',
        'avg_last50_global', 'avg_last50_survived',
        'avg_max_pop', 'avg_steps_alive',
        'total_moves',
        'std_final_global', 'std_final_survived',
        'std_last50_global', 'std_last50_survived',
        'std_moves',
        'n_extinctions', 'extinction_rate',
        'avg_prob_N', 'avg_prob_S', 'avg_prob_E', 'avg_prob_W', 'avg_prob_Stay',
        *NEW_SUMMARY_COLUMNS,
    ]
    df = _to_numeric(df, numeric_cols)

    if 'n_extinctions' not in df.columns and 'extinction_rate' in df.columns:
        df['n_extinctions'] = (df['extinction_rate'] * NUM_EPISODES).round().astype('Int64')
    if 'extinction_rate' not in df.columns and 'n_extinctions' in df.columns:
        df['extinction_rate'] = df['n_extinctions'] / NUM_EPISODES

    parsed = df['mode'].apply(parse_mode)
    df = pd.concat([df, parsed], axis=1)
    df['layer_label'] = np.where(df['layer'].fillna(0).astype(int).eq(0), 'Rule/Random', 'L' + df['layer'].fillna(0).astype(int).astype(str))
    return df

def load_timeseries(path=TS_CSV):
    if not Path(path).exists():
        print(f'{path} not found. Timeseries plots will be skipped.')
        return None

    ts = pd.read_csv(path)
    if Path(path).resolve() == TS_CSV.resolve():
        llm_ts_frames = []
        for model in LLM_MODELS:
            for treatment, (_, ts_path) in LLM_FILES[model].items():
                if not ts_path.exists():
                    print(f'LLM timeseries missing: {ts_path.name}')
                    continue
                llm_part = pd.read_csv(ts_path)
                llm_part['mode'] = f'LLM-{model}-{treatment}'
                llm_ts_frames.append(llm_part)
                print(f'LLM timeseries appended: {ts_path}')
        if llm_ts_frames:
            ts = pd.concat([ts, *llm_ts_frames], ignore_index=True)
    required = {'scenario', 'layer', 'mode', 'step', 'avg_pop_step', 'avg_prop_step'}
    missing = required - set(ts.columns)
    if missing:
        raise ValueError(f'Timeseries CSV missing columns: {sorted(missing)}')

    ts['mode'] = ts['mode'].replace({'Random_Random': 'Random'})
    ts = _to_numeric(ts, [
        'layer', 'step',
        'avg_pop_step',
        'avg_prop_step', 'std_prop_step', 'n_prop_samples',
        'avg_beta_step', 'std_beta_step', 'n_beta_samples',
        'n_pop_samples',
        *NEW_TIMESERIES_COLUMNS,
    ])
    ts = ts.dropna(subset=['scenario', 'mode', 'layer', 'step'])
    ts['layer'] = ts['layer'].astype(int)
    ts['step'] = ts['step'].astype(int)

    if 'n_pop_samples' in ts.columns:
        ts['global_avg_pop'] = ts['avg_pop_step'] * ts['n_pop_samples'] / NUM_EPISODES
        ts['survival_rate'] = ts['n_pop_samples'] / NUM_EPISODES

    parsed = ts['mode'].apply(parse_mode)
    return pd.concat([ts, parsed], axis=1)

df = load_summary()
ts_df = load_timeseries()

available_new_summary = [c for c in NEW_SUMMARY_COLUMNS if c in df.columns]
missing_new_summary = [c for c in NEW_SUMMARY_COLUMNS if c not in df.columns]
available_new_ts = [] if ts_df is None else [c for c in NEW_TIMESERIES_COLUMNS if c in ts_df.columns]
missing_new_ts = NEW_TIMESERIES_COLUMNS if ts_df is None else [c for c in NEW_TIMESERIES_COLUMNS if c not in ts_df.columns]

print('Summary loaded')
print(f'  rows: {len(df)}')
print(f'  scenarios: {sorted(df.scenario.dropna().unique())}')
print(f"  modes: {df['mode'].nunique()}")
print(f'  timeseries rows: {0 if ts_df is None else len(ts_df)}')
print(f'  new summary metric columns available: {len(available_new_summary)}/{len(NEW_SUMMARY_COLUMNS)}')
print(f'  new timeseries metric columns available: {len(available_new_ts)}/{len(NEW_TIMESERIES_COLUMNS)}')
if missing_new_summary or missing_new_ts:
    print('  Note: regenerate the CSVs with run_analysis.py / run_analysis_llm.py to populate missing behavioural metrics.')

display(df.head())

In [ ]:
MODE_ORDER = [
    'Random',
    'Rule-Random/Fix', 'Rule-Random/Random', 'Rule-Random/Evo', 'Rule-Random/Learn',
    'Rule-Fixed/Fix', 'Rule-Fixed/Evo', 'Rule-Fixed/Learn',
    'MLP-Random/Fix', 'MLP-Random/Random', 'MLP-Random/Evo', 'MLP-Random/Learn',
    'MLP-Offline/Fix', 'MLP-Offline/Evo', 'MLP-Offline/Learn',
    'MLP-DAgger/Fix', 'MLP-DAgger/Evo', 'MLP-DAgger/Learn',
    'MLP-EvoTop5/Fix', 'MLP-EvoTop5/Evo', 'MLP-EvoTop5/Learn',
    'MLP-EvoTop5-DAgger/Fix', 'MLP-EvoTop5-DAgger/Evo', 'MLP-EvoTop5-DAgger/Learn',
    'LLM-gpt-4o-mini-Goal', 'LLM-gpt-4o-mini-NoGoal',
    'LLM-gpt-5-nano-Goal', 'LLM-gpt-5-nano-NoGoal',
    'LLM-gpt-5-mini-Goal', 'LLM-gpt-5-mini-NoGoal',
]

# Hierarchical plasma scheme shared with the paper and thesis figures:
# nearby shades identify variants within the same policy-origin family.
# Each LLM model uses a separate plasma region; prompts differ by shade.
MODE_COLORS = {
    'Random': '#888888',

    'Rule-Random/Fix': '#CD4A76',
    'Rule-Random/Random': '#D9586A',
    'Rule-Random/Evo': '#E3685F',
    'Rule-Random/Learn': '#EB7655',

    'Rule-Fixed/Fix': '#FDAE32',
    'Rule-Fixed/Evo': '#FDC229',
    'Rule-Fixed/Learn': '#FBD724',

    'MLP-Random/Fix': '#2F0596',
    'MLP-Random/Random': '#46039F',
    'MLP-Random/Evo': '#5901A5',
    'MLP-Random/Learn': '#6E00A8',

    'MLP-Offline/Fix': '#9613A1',
    'MLP-Offline/Evo': '#A72197',
    'MLP-Offline/Learn': '#B6308B',
    'MLP-DAgger/Fix': '#9613A1',
    'MLP-DAgger/Evo': '#A72197',
    'MLP-DAgger/Learn': '#B6308B',
    'MLP-EvoTop5/Fix': '#9613A1',
    'MLP-EvoTop5/Evo': '#A72197',
    'MLP-EvoTop5/Learn': '#B6308B',
    'MLP-EvoTop5-DAgger/Fix': '#9613A1',
    'MLP-EvoTop5-DAgger/Evo': '#A72197',
    'MLP-EvoTop5-DAgger/Learn': '#B6308B',

    'LLM-gpt-4o-mini-Goal': '#38049A',
    'LLM-gpt-4o-mini-NoGoal': '#5102A3',
    'LLM-gpt-5-nano-Goal': '#AB2494',
    'LLM-gpt-5-nano-NoGoal': '#BC3587',
    'LLM-gpt-5-mini-Goal': '#F58B47',
    'LLM-gpt-5-mini-NoGoal': '#FCA338',
}

LLM_MODE_LABELS = {
    'LLM-gpt-4o-mini-Goal': 'GPT-4o mini — Goal',
    'LLM-gpt-4o-mini-NoGoal': 'GPT-4o mini — No goal',
    'LLM-gpt-5-mini-Goal': 'GPT-5 mini — Goal',
    'LLM-gpt-5-mini-NoGoal': 'GPT-5 mini — No goal',
    'LLM-gpt-5-nano-Goal': 'GPT-5 nano — Goal',
    'LLM-gpt-5-nano-NoGoal': 'GPT-5 nano — No goal',
}
LLM_CMAP = LinearSegmentedColormap.from_list('llm_rose', ['#FBE9EA', '#E66B73', '#6D071A'])

def ordered_modes(values):
    values = list(pd.Series(values).dropna().unique())
    return [m for m in MODE_ORDER if m in values] + sorted([m for m in values if m not in MODE_ORDER])

# Original notebook style: compact V0, V1, ... labels in legends.
_modes_all = ordered_modes(df['mode'].dropna().unique())
MODE_LABEL = {mode: LLM_MODE_LABELS.get(mode, f'V{i}') for i, mode in enumerate(_modes_all)}
print('Variant labels assigned:')
for _mode, _label in MODE_LABEL.items():
    print(f'  {_label}: {_mode}')

def metric_table(data=df, metric='avg_last50_global', top_n=15, ascending=False):
    cols = ['scenario', 'family', 'training', 'adaptation', 'layer', 'mode', metric, 'extinction_rate', 'avg_steps_alive']
    available = [c for c in cols if c in data.columns]
    out = data[available].sort_values(metric, ascending=ascending).head(top_n).copy()
    return out

display(metric_table(metric='avg_last50_global', top_n=20))


In [ ]:
# Summary plots by scenario, split by family
# Rule-Based includes the pure Random baseline, as in the original paper figures.
plot_df = df.copy()
plot_df['mode_label'] = plot_df['mode'].map(MODE_LABEL).fillna(plot_df['mode'])

family_groups = [
    ('Rule-Based + Random', plot_df[plot_df['family'].isin(['Random', 'Rule'])].copy()),
    ('MLP', plot_df[plot_df['family'].eq('MLP')].copy()),
    ('LLM', plot_df[plot_df['family'].eq('LLM')].copy()),
]

def plot_family_metric_bars(metric_group, group_title):
    available_metrics = [(metric, ylabel) for metric, ylabel in metric_group if metric in plot_df.columns]
    if not available_metrics:
        print(f'No columns available for {group_title}.')
        return

    for family_title, family_df in family_groups:
        if family_df.empty:
            print(f'No rows available for {family_title}.')
            continue

        family_modes = ordered_modes(family_df['mode'])
        family_palette = {mode: MODE_COLORS.get(mode, '#7f8c8d') for mode in family_modes}

        for metric, ylabel in available_metrics:
            data = family_df.dropna(subset=[metric])
            if data.empty:
                print(f'{family_title}: no values available for {metric}.')
                continue
            plt.figure(figsize=(13, 5))
            ax = sns.barplot(
                data=data,
                x='scenario',
                y=metric,
                hue='mode',
                hue_order=family_modes,
                palette=family_palette,
                **BARPLOT_KW,
            )
            ax.set_title(f'{family_title}: {ylabel}', fontweight='bold')
            ax.set_xlabel('Scenario')
            ax.set_ylabel(ylabel)
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(
                handles,
                [MODE_LABEL.get(label, label) for label in labels],
                title='Variant',
                bbox_to_anchor=(1.02, 1),
                loc='upper left',
                framealpha=0.9,
            )
            sns.despine()
            plt.tight_layout()
            plt.show()

plot_family_metric_bars(SUMMARY_METRICS, 'core summary metrics')
plot_family_metric_bars(BEHAVIORAL_SUMMARY_METRICS, 'behavioural summary metrics')

In [ ]:
# MLP layer comparison
mlp = df[(df['family'] == 'MLP') & (df['layer'] > 0)].copy()

if mlp.empty:
    print('No MLP rows available.')
else:
    for metric, ylabel in SUMMARY_METRICS + BEHAVIORAL_SUMMARY_METRICS:
        if metric not in mlp.columns:
            continue
        data = mlp.dropna(subset=[metric])
        if data.empty:
            continue
        g = sns.relplot(
            data=data,
            kind='line',
            x='layer',
            y=metric,
            hue='mode',
            hue_order=ordered_modes(data['mode']),
            col='scenario',
            col_wrap=3,
            marker='o',
            palette=MODE_COLORS,
            height=3.5,
            aspect=1.2,
            facet_kws={'sharey': False},
        )
        g.set_axis_labels('Hidden layers', ylabel)
        g.set_titles('{col_name}')
        if g._legend is not None:
            for text in g._legend.texts:
                text.set_text(MODE_LABEL.get(text.get_text(), text.get_text()))
            g._legend.set_title('Mode')
        g.fig.suptitle(f'MLP layer sensitivity: {ylabel}', y=1.03, fontweight='bold')
        plt.show()

In [ ]:
# Best configuration per scenario
metric = 'avg_last50_global'
rank_df = df.dropna(subset=[metric]).copy()
rank_df['rank_in_scenario'] = rank_df.groupby('scenario')[metric].rank(method='dense', ascending=False)

best = (
    rank_df[rank_df['rank_in_scenario'] <= 5]
    .sort_values(['scenario', 'rank_in_scenario', metric], ascending=[True, True, False])
    [['scenario', 'rank_in_scenario', 'mode', 'layer', metric, 'avg_steps_alive', 'extinction_rate']]
)
display(best)

for family_title, family_df in [
    ('Rule-Based + Random', rank_df[rank_df['family'].isin(['Random', 'Rule'])].copy()),
    ('MLP', rank_df[rank_df['family'].eq('MLP')].copy()),
    ('LLM', rank_df[rank_df['family'].eq('LLM')].copy()),
]:
    if family_df.empty:
        continue
    pivot = family_df.pivot_table(index='mode', columns='scenario', values=metric, aggfunc='max')
    pivot = pivot.reindex(ordered_modes(pivot.index))

    plt.figure(figsize=(10, max(4, 0.45 * len(pivot))))
    heatmap_cmap = LLM_CMAP if family_title == 'LLM' else 'viridis'
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap=heatmap_cmap, linewidths=0.5)
    plt.title(f'{family_title}: best {metric} by mode and scenario', fontweight='bold')
    plt.xlabel('Scenario')
    plt.ylabel('Mode')
    plt.yticks(
        ticks=np.arange(len(pivot.index)) + 0.5,
        labels=[MODE_LABEL.get(m, m) for m in pivot.index],
        rotation=0,
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# Scatter: survival steps vs maximum population
scatter_df = df.dropna(subset=['avg_max_pop', 'avg_steps_alive']).copy()

if scatter_df.empty:
    print('No data available for the scatter plot.')
else:
    for family_title, family_df in [
        ('Rule-Based + Random', scatter_df[scatter_df['family'].isin(['Random', 'Rule'])].copy()),
        ('MLP', scatter_df[scatter_df['family'].eq('MLP')].copy()),
        ('LLM', scatter_df[scatter_df['family'].eq('LLM')].copy()),
    ]:
        if family_df.empty:
            continue
        family_modes = ordered_modes(family_df['mode'])
        family_palette = {mode: MODE_COLORS.get(mode, '#7f8c8d') for mode in family_modes}
        g = sns.relplot(
            data=family_df,
            x='avg_max_pop',
            y='avg_steps_alive',
            hue='mode',
            hue_order=family_modes,
            style='layer',
            col='scenario',
            col_wrap=3,
            s=85,
            alpha=0.85,
            palette=family_palette,
            height=3.8,
            aspect=1.2,
            facet_kws={'sharex': False, 'sharey': False},
        )
        g.set_axis_labels('Mean maximum population', 'Mean survival steps')
        g.set_titles('{col_name}')
        if g._legend is not None:
            for text in g._legend.texts:
                text.set_text(MODE_LABEL.get(text.get_text(), text.get_text()))
            g._legend.set_title('Variant / layer')
        g.fig.suptitle(f'{family_title}: survival vs peak population', y=1.03, fontweight='bold')
        plt.show()


In [ ]:
# Timeseries explorer
if ts_df is None or ts_df.empty:
    print('No timeseries data available.')
else:
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output

        scenarios = sorted(ts_df['scenario'].unique())
        modes = ordered_modes(ts_df['mode'])
        layers = sorted(ts_df['layer'].unique())

        scenario_w = widgets.Dropdown(options=scenarios, value=scenarios[0], description='Scenario')
        modes_w = widgets.SelectMultiple(
            options=[(MODE_LABEL.get(m, m), m) for m in modes],
            value=tuple(modes[: min(4, len(modes))]),
            description='Modes',
            rows=min(12, len(modes)),
        )
        layers_w = widgets.SelectMultiple(options=layers, value=tuple(layers[: min(3, len(layers))]), description='Layers')

        _metric_options = [(label, metric) for label, metric in TIMESERIES_METRICS if metric in ts_df.columns]
        if not _metric_options:
            raise ValueError('No supported timeseries metrics are available in the loaded CSVs.')

        missing_ts_metrics = [metric for _, metric in TIMESERIES_METRICS if metric not in ts_df.columns]
        if missing_ts_metrics:
            print('Missing timeseries metrics:', ', '.join(missing_ts_metrics))
            print('Regenerate the CSVs to enable the behavioural metrics.')

        metric_w = widgets.Dropdown(
            options=_metric_options,
            value=_metric_options[0][1],
            description='Metric',
        )
        max_step_w = widgets.IntSlider(value=min(500, int(ts_df['step'].max())), min=10, max=int(ts_df['step'].max()), step=10, description='Max step')
        out = widgets.Output()

        _STD_COL = {
            'avg_prop_step': 'std_prop_step',
            'avg_beta_step': 'std_beta_step',
        }

        _METRIC_LABEL = {metric: label for label, metric in TIMESERIES_METRICS}

        def draw(*_):
            with out:
                clear_output(wait=True)
                metric = metric_w.value
                if metric not in ts_df.columns:
                    print(f'{metric} is not available in this CSV.')
                    return
                data = ts_df[
                    (ts_df['scenario'] == scenario_w.value)
                    & (ts_df['mode'].isin(list(modes_w.value)))
                    & (ts_df['layer'].isin(list(layers_w.value)))
                    & (ts_df['step'] <= max_step_w.value)
                ].copy()
                if data.empty:
                    print('No rows match the current filters.')
                    return

                data['mode_label'] = data['mode'].map(MODE_LABEL).fillna(data['mode'])
                ts_palette = {
                    MODE_LABEL.get(mode, mode): MODE_COLORS.get(mode, '#7f8c8d')
                    for mode in data['mode'].dropna().unique()
                }

                std_col = _STD_COL.get(metric)
                ylabel = _METRIC_LABEL.get(metric, metric)
                plt.figure(figsize=(11, 5.5))
                ax = plt.gca()
                if std_col and std_col in data.columns:
                    for (mode_lbl, layer_val), grp in data.groupby(['mode_label', 'layer']):
                        grp = grp.sort_values('step')
                        color = ts_palette.get(mode_lbl, '#7f8c8d')
                        ax.plot(grp['step'], grp[metric], label=f'{mode_lbl} L{layer_val}', color=color, linewidth=1.7)
                        ax.fill_between(
                            grp['step'],
                            grp[metric] - grp[std_col].fillna(0),
                            grp[metric] + grp[std_col].fillna(0),
                            alpha=0.15,
                            color=color,
                        )
                    ax.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left')
                else:
                    sns.lineplot(data=data, x='step', y=metric, hue='mode_label', style='layer', palette=ts_palette, linewidth=1.7, ax=ax)
                    ax.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left')

                ax.set_title(f"{scenario_w.value}: {ylabel}", fontweight='bold')
                ax.set_xlabel('Step')
                ax.set_ylabel(ylabel)
                plt.tight_layout()
                plt.show()

        for w in [scenario_w, modes_w, layers_w, metric_w, max_step_w]:
            w.observe(draw, names='value')

        controls = widgets.VBox([
            widgets.HBox([scenario_w, metric_w, max_step_w]),
            widgets.HBox([modes_w, layers_w]),
        ])
        display(controls, out)
        draw()

    except ImportError:
        print('ipywidgets is not installed; showing a static example instead.')
        metric = next((m for _, m in TIMESERIES_METRICS if m in ts_df.columns), 'avg_pop_step')
        ylabel = dict((m, label) for label, m in TIMESERIES_METRICS).get(metric, metric)
        example = ts_df[
            (ts_df['scenario'] == sorted(ts_df['scenario'].unique())[0])
            & (ts_df['mode'].isin(ordered_modes(ts_df['mode'])[:4]))
            & (ts_df['step'] <= 500)
        ].copy()
        example['mode_label'] = example['mode'].map(MODE_LABEL).fillna(example['mode'])
        ts_palette = {
            MODE_LABEL.get(mode, mode): MODE_COLORS.get(mode, '#7f8c8d')
            for mode in example['mode'].dropna().unique()
        }
        plt.figure(figsize=(11, 5.5))
        sns.lineplot(data=example, x='step', y=metric, hue='mode_label', style='layer', palette=ts_palette)
        plt.ylabel(ylabel)
        plt.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        plt.show()